# Demo 14. Power and Inverse Iteration

**Standard 8** (eigenvalue computation), Week 6.

The power method finds the eigenvalue of largest magnitude by repeated multiplication by $A$. A shift and an inverse turn the same idea into a method that finds any target eigenvalue. This demo tracks the convergence of both against a matrix with a known spectrum.

In [ ]:
# --- Colab / Jupyter setup ------------------------------------------------
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider

%matplotlib inline
plt.rcParams["figure.figsize"] = (9, 5)

## Why it works

Expand the start vector in an eigenbasis, $x_0 = \sum_i c_i v_i$. Then

$$ A^k x_0 = \sum_i c_i \lambda_i^k v_i = \lambda_1^k\Big(c_1 v_1 + \sum_{i\ge 2} c_i (\lambda_i/\lambda_1)^k v_i\Big). $$

If $|\lambda_1| > |\lambda_i|$ for $i \ge 2$, the trailing terms decay and the iterate aligns with $v_1$; the error falls like $|\lambda_2/\lambda_1|^k$. Applying the method to $(A-\sigma I)^{-1}$ makes the eigenvalue of $A$ nearest $\sigma$ dominant, which isolates it.

In [ ]:
# ---------------------------------------------------------------------------
# Power iteration: converge to the dominant eigenvector
# ---------------------------------------------------------------------------
# Repeatedly applying A and normalising drives a generic starting vector toward
# the eigenvector of the largest-magnitude eigenvalue. The Rayleigh quotient
# x^T A x / x^T x then estimates that eigenvalue. The error decreases like
# |lambda_2 / lambda_1|^k, so a small gap between the top two eigenvalues gives
# slow convergence.

def power_iteration(A, n_iter, x0=None):
    """Power iteration. Returns the eigenvalue estimate per step (via the
    Rayleigh quotient) and the final vector."""
    A = np.asarray(A, float)
    n = A.shape[0]
    x = np.ones(n) if x0 is None else x0.copy()
    x = x / np.linalg.norm(x)
    estimates = []
    for _ in range(n_iter):
        y = A @ x
        x = y / np.linalg.norm(y)
        estimates.append(float(x @ A @ x))     # Rayleigh quotient
    return np.array(estimates), x

In [ ]:
# ---------------------------------------------------------------------------
# Inverse iteration with a shift: target the eigenvalue nearest sigma
# ---------------------------------------------------------------------------
# Power iteration on (A - sigma I)^{-1} converges to the eigenvector whose
# eigenvalue is closest to the shift sigma, because that eigenvalue of A becomes
# the largest-magnitude eigenvalue of the shifted inverse. Choosing sigma near a
# target eigenvalue isolates it.

def inverse_iteration(A, sigma, n_iter, x0=None):
    """Inverse iteration with shift sigma. Returns eigenvalue estimates per step
    and the final vector. Solves a shifted system each step instead of forming
    an explicit inverse."""
    A = np.asarray(A, float)
    n = A.shape[0]
    M = A - sigma * np.eye(n)
    x = np.ones(n) if x0 is None else x0.copy()
    x = x / np.linalg.norm(x)
    estimates = []
    for _ in range(n_iter):
        y = np.linalg.solve(M, x)               # (A - sigma I) y = x
        x = y / np.linalg.norm(y)
        estimates.append(float(x @ A @ x))
    return np.array(estimates), x

In [ ]:
# ---------------------------------------------------------------------------
# A symmetric matrix with known eigenvalues, so we can measure the error
# ---------------------------------------------------------------------------
def make_symmetric(eigs, seed=0):
    """Build a symmetric matrix with the prescribed eigenvalues via a random
    orthogonal similarity transform Q diag(eigs) Q^T."""
    eigs = np.asarray(eigs, float)
    n = len(eigs)
    rng = np.random.default_rng(seed)
    Q, _ = np.linalg.qr(rng.standard_normal((n, n)))
    return Q @ np.diag(eigs) @ Q.T

EIGS = np.array([10.0, 6.0, 3.0, 1.0])   # known spectrum for the demo
A_demo = make_symmetric(EIGS)

def show_power(n_iter=25):
    """Power iteration converging to the largest eigenvalue (10)."""
    est, _ = power_iteration(A_demo, n_iter)
    target = EIGS.max()
    plt.figure()
    plt.semilogy(np.abs(est - target) + 1e-18, "r.-")
    plt.xlabel("iteration"); plt.ylabel("|estimate - lambda_max|")
    plt.title(f"Power iteration -> {target:g}  (rate |l2/l1| = {EIGS[1]/EIGS[0]:.2f})")
    plt.show()
    print("final eigenvalue estimate:", est[-1])

def show_inverse(sigma=5.0, n_iter=15):
    """Inverse iteration finding the eigenvalue nearest the shift sigma."""
    est, _ = inverse_iteration(A_demo, sigma, n_iter)
    nearest = EIGS[np.argmin(np.abs(EIGS - sigma))]
    plt.figure()
    plt.semilogy(np.abs(est - nearest) + 1e-18, "g.-")
    plt.xlabel("iteration"); plt.ylabel("|estimate - nearest eigenvalue|")
    plt.title(f"Inverse iteration, shift sigma={sigma:g} -> nearest eigenvalue {nearest:g}")
    plt.show()
    print(f"spectrum = {EIGS},  shift = {sigma},  converged to {est[-1]:.6f}")

show_power(25)
show_inverse(5.0, 15)

In [ ]:
# ---------------------------------------------------------------------------
# Interactive: move the shift to select which eigenvalue inverse iteration finds
# ---------------------------------------------------------------------------
interact(show_power, n_iter=IntSlider(value=25, min=5, max=80, step=5, description="# iters"));
interact(
    show_inverse,
    sigma=FloatSlider(value=5.0, min=0.0, max=11.0, step=0.25, description="shift sigma"),
    n_iter=IntSlider(value=15, min=3, max=40, step=1, description="# iters"),
);

## Summary

- Power iteration converges to the dominant eigenpair; the Rayleigh quotient estimates the eigenvalue.
- The convergence rate is $|\lambda_2/\lambda_1|^k$, so a small spectral gap converges slowly.
- Inverse iteration with shift $\sigma$ converges to the eigenvalue nearest $\sigma$, solving a shifted linear system each step.
- Moving the shift selects the target eigenvalue, which underlies practical eigenvalue algorithms.